# taar-pocketome — pipeline walkthrough

This notebook is the entry point for anyone new to this codebase. It walks through the three pipeline stages described in the top-level `README.md`:

1. **Stage 1** (`pipeline/`) — parse raw pocket-search output, cluster local pockets into global pocket IDs, classify orthosteric pockets.
2. **Stage 2** (`pipeline/`) — post-processing: largest/orthosteric/transient classification, median volumes, all distribution plots.
3. **Stage 3** (`global_id/`) — the global-ID extension: apo↔holo global-ID matching, replicate concordance, and the paper figures.

**No data ships with this repo.** Every cell below that reads real data is marked ⚠️ and will raise a clear `FileNotFoundError` until you've dropped your own dataset into `input/` (see the *Setup* section). Read the markdown even if you can't run the cell — it explains what each stage expects and produces.

Data availability: **TODO — link to the published data repository / DOI**.

## Setup

Run this notebook from a Python environment created from `environment.yml` (see `README.md`).

`config.py` at the repo root resolves `PROJECT_ROOT` from the `TAAR_ROOT` environment variable, falling back to the current working directory. Jupyter's working directory is normally the directory the notebook lives in (`notebooks/`), so we point `TAAR_ROOT` at the repo root explicitly below rather than relying on the fallback.

In [ ]:
import os
import sys

REPO_ROOT = os.path.dirname(os.getcwd())  # notebooks/ -> repo root
os.environ["TAAR_ROOT"] = REPO_ROOT

for sub in ["", "pipeline", "global_id", "scripts"]:
    p = os.path.join(REPO_ROOT, sub)
    if p not in sys.path:
        sys.path.insert(0, p)

from config import PROJECT_ROOT, INPUT_DIR, OUTPUT_DIR, APO_STRUCTURES_DIR, HOLO_STRUCTURES_DIR, HOLO_BASE_DIR, META_ANALYSIS_DIR

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_DIR:   ", INPUT_DIR)
print("OUTPUT_DIR:  ", OUTPUT_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)

has_data = os.path.isdir(APO_STRUCTURES_DIR) or os.path.isdir(HOLO_STRUCTURES_DIR)
print("Data present in input/:", has_data)
if not has_data:
    print("\n\u26a0️  No data found under input/. Cells that read real data will not run until you")
    print("   populate input/apo_structures/ and input/holo_structures/ — see README.md for the")
    print("   expected layout, and the data-availability link at the top of this notebook.")

## Stage 1 — parsing & global-ID clustering

`comparative_study.run_analysis(prj_ls, saving_loc)` is the core Stage-1 entry point. It:

1. Instantiates `meta_analysis_class.MetaAnalysis` over the given list of pocket-search project directories.
2. Parses the raw mdpocket/ATClus output (`pock_file_parser`).
3. Interpolates pocket volume across frames.
4. Assigns global cluster (global pocket) IDs via voxel-IoU clustering.

`pipeline/comp_with_sel.py` wraps this in a CLI (`--within-rep`, `--apo-vs-holo`, `--within-gene`, `--across-genes`) for cluster/SLURM use; the cell below calls the function directly instead, which is easier to inspect interactively.

⚠️ **Needs data.** `prj_ls` below is a placeholder — replace it with your own pocket-search project directories under `input/`.

In [ ]:
from comparative_study import run_analysis

# Replace with real project directories, e.g. everything under
# HOLO_STRUCTURES_DIR/results/<pdb_id>/<rep>/pockets_dens
prj_ls = []  # <- fill in

if prj_ls:
    summary_df = run_analysis(prj_ls, saving_loc=META_ANALYSIS_DIR)
else:
    print("prj_ls is empty — fill it in with real pocket-search directories before running Stage 1.")

## Stage 2 — post-processing & visualizations

`complete_pocket_analysis.run_complete_pocket_analysis(...)` reads Stage 1's `summary_df_3d_coords.csv`, then:

- Classifies the largest pocket per experiment.
- Applies the ligand-based orthosteric filter (`orthosteric_filter_ligand_based.py`, single source of truth for `is_orthosteric`; default 5 Å from the holo ligand centroid).
- Flags transient vs. stable pockets (`consecutive_zeros_transiency.py`).
- Generates the full set of distribution plots via `visualizations.py`.

It writes `pocket_analysis_summary.csv` and `orthosteric_perframe_volumes.csv`, both of which Stage 3 reads.

⚠️ **Needs Stage 1's output** (`summary_df_3d_coords.csv` under `META_ANALYSIS_DIR`) to already exist.

In [ ]:
from complete_pocket_analysis import run_complete_pocket_analysis

data_path = os.path.join(META_ANALYSIS_DIR, "summary_df_3d_coords.csv")

if os.path.exists(data_path):
    results = run_complete_pocket_analysis(
        saving_loc=META_ANALYSIS_DIR,
        data_path=data_path,
        holo_base=HOLO_BASE_DIR,
        distance_threshold=5.0,
        export_pdfs=True,
        run_3d_heatmap=True,
        run_apo_holo_comparison=True,
        run_violin_plots=True,
    )
else:
    print(f"Stage 1 output not found at {data_path} — run Stage 1 first.")

## Stage 3 — global pocket IDs

A **global pocket ID** is a dataset-scoped identifier that makes the same pocket comparable across MD replicates and across the apo/holo split — each raw run otherwise produces its own independent local numbering.

### 3a. Per-state global-ID runs

`global_id/global_id_wrapper_paper.py` re-runs the Stage-1 driver body once per state (`apo`, `holo`, `both`, or `combined`), each into its own non-colliding output directory (`output/global_ID_{state}_only/`). It's a CLI script (`--state {apo,holo,both,combined}`), so it's run as a subprocess here rather than imported.

In [ ]:
import subprocess

GLOBAL_ID_DIR = os.path.join(REPO_ROOT, "global_id")

if has_data:
    for state in ["apo", "holo"]:
        subprocess.run(
            [sys.executable, "global_id_wrapper_paper.py", "--state", state],
            cwd=GLOBAL_ID_DIR,
            check=True,
        )
else:
    print("No data under input/ — skipping. See the Setup cell above.")

### 3b. Matching apo global IDs onto holo global IDs

`global_id/gid_state_mapping.py` recovers the apo↔holo correspondence geometrically: it takes the centroid of every global pocket in each state and does bidirectional nearest-centroid matching (both states are already superposed into one coordinate frame, so raw Euclidean centroid distance is meaningful). Output goes to `output/apo_holo_global_id_mapping/`.

In [ ]:
if has_data:
    subprocess.run([sys.executable, "gid_state_mapping.py"], cwd=GLOBAL_ID_DIR, check=True)
else:
    print("No data under input/ — skipping.")

### 3c. Replicate concordance — the argument that global IDs are reliable

`global_id/gid_replicate_analysis_wrapper.py` answers three questions about a set of replicate MD runs (same PDB, same state, multiple replicates):

- **Descriptive**: given the global IDs the pipeline assigned, how consistent is the pocketome across replicates? (counts, occupancy spectrum, Jaccard vs. a permutation null, rarefaction)
- **GID spread**: do local pockets sharing a global ID actually sit closer to each other than to members of other global IDs? (silhouette on pocket centroids — an independent check, since the clustering itself works on voxel overlap, not centroid distance)
- **Threshold sweep**: how much of the partition depends on the clustering parameters (`iou_thresh`, `dilation_radius`)? Re-clusters over a parameter grid and counts how many pockets change occupancy class.

`global_id/replicate_figures.py` renders the corresponding figure (count collapse, occupancy, presence matrix; supplementary: Jaccard heatmap, rarefaction). `global_id/pymol_ambiguous_pockets.py` writes a PyMOL scene per pocket the threshold sweep flagged as unstable, for visual inspection (needs PyMOL, installed separately, to open the output).

In [ ]:
if has_data:
    subprocess.run([sys.executable, "gid_replicate_analysis_wrapper.py"], cwd=GLOBAL_ID_DIR, check=True)
    subprocess.run([sys.executable, "replicate_figures.py"], cwd=GLOBAL_ID_DIR, check=True)
else:
    print("No data under input/ — skipping.")

### 3d. Paper summary table & cross-experiment heatmap

- `global_id/summary_global_ids_paper.py` — one row per global ID over the combined apo+holo run, built from `pocket_comparison_table.csv`.
- `global_id/global_id_heatmap.py` — rows = global ID, columns = every experiment (gene → state → replicate), cell = median pocket volume; a dot marks fusion (≥2 local pockets collapsed into one global ID in that experiment).

In [ ]:
if has_data:
    subprocess.run([sys.executable, "summary_global_ids_paper.py"], cwd=GLOBAL_ID_DIR, check=True)
    subprocess.run([sys.executable, "global_id_heatmap.py"], cwd=GLOBAL_ID_DIR, check=True)
else:
    print("No data under input/ — skipping.")

### 3e. Blender render prep (optional, needs Blender + Molecular Nodes)

`global_id/blender_global_id_prep.py` picks, per global pocket, the MD frame closest to its median volume and writes it as a Molecular-Nodes-ready PDB. `global_id/calibrate_pocket_radii.py` then solves the sphere radius that reproduces each pocket's *reported* volume — drawing pockets at their raw alpha-sphere radius overstates their size by 3–5x, since an alpha sphere reaches out to the four protein atoms it touches. Both scripts only prepare inputs; the actual render happens inside Blender, outside this notebook.

In [ ]:
if has_data:
    subprocess.run([sys.executable, "blender_global_id_prep.py"], cwd=GLOBAL_ID_DIR, check=True)
    subprocess.run([sys.executable, "calibrate_pocket_radii.py"], cwd=GLOBAL_ID_DIR, check=True)
else:
    print("No data under input/ — skipping.")

## Where to go next

- `README.md` at the repo root has the full script-by-script reference, the `input/`/`output/` layout, and the external-tool install notes (PyMOL, Blender + Molecular Nodes, `atclus`).
- `reference_data/` holds the small lookup tables (GPCRdb residue numbering, gene-name map) the pipeline needs to run — not experimental output, so they're checked into the repo rather than published with the data.
- `archive/scripts/` holds superseded/one-off scripts kept for reference only; nothing in `pipeline/` or `global_id/` imports from there.